In [1]:
from datasets import load_dataset

In [2]:
d = load_dataset("brimmann2/squad-v2-sampled")

In [4]:
%cd /home/brimmann/works/small-xrag

/home/brimmann/works/small-xrag


In [5]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
from imported_code.modeling_sfr import SFR

In [6]:
retriever_name = "Salesforce/SFR-Embedding-Mistral"
retriever = SFR.from_pretrained(retriever_name,torch_dtype = torch.bfloat16).eval().to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
from transformers import AutoTokenizer
retriever_tokenizer = AutoTokenizer.from_pretrained(retriever_name)

In [10]:
def embed(context):
    inputs = retriever_tokenizer(
        context,
        max_length=180,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        embeds = retriever.get_doc_embedding(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
        )
    return embeds

In [11]:
all_embeds_train = []
for i in range(len(d["train"])):
    out = embed(d["train"][i]["context"]).squeeze(0).cpu().tolist()
    all_embeds_train.append(out)

In [16]:
updated_ds = d["train"].add_column("embeddings", all_embeds_train)

In [27]:
updated_ds

Dataset({
    features: ['gem_id', 'id', 'title', 'context', 'question', 'target', 'references', 'answers', 'embeddings'],
    num_rows: 1000
})

In [21]:
all_embeds_dev = []
for i in range(len(d["dev"])):
    out = embed(d["dev"][i]["context"]).squeeze(0).cpu().tolist()
    all_embeds_dev.append(out)

In [22]:
len(all_embeds_dev)

250

In [28]:
updated_ds_dev = d["dev"].add_column("embeddings", all_embeds_dev)

In [24]:
all_embeds_test = []
for i in range(len(d["test"])):
    out = embed(d["test"][i]["context"]).squeeze(0).cpu().tolist()
    all_embeds_test.append(out)

In [29]:
updated_ds_test = d["test"].add_column("embeddings", all_embeds_test)

In [30]:
d["train"] = updated_ds
d["dev"] = updated_ds_dev
d["test"] = updated_ds_test

In [33]:
d

DatasetDict({
    train: Dataset({
        features: ['gem_id', 'id', 'title', 'context', 'question', 'target', 'references', 'answers', 'embeddings'],
        num_rows: 1000
    })
    dev: Dataset({
        features: ['gem_id', 'id', 'title', 'context', 'question', 'target', 'references', 'answers', 'embeddings'],
        num_rows: 250
    })
    test: Dataset({
        features: ['gem_id', 'id', 'title', 'context', 'question', 'target', 'references', 'answers', 'embeddings'],
        num_rows: 250
    })
})

In [34]:
d.push_to_hub(
    "brimmann2/squad-v2-sampled",
    commit_message="Add column embeddings"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/brimmann2/squad-v2-sampled/commit/08dfe281e304edd15c7c6f094f643c2b02149f36', commit_message='Add column embeddings', commit_description='', oid='08dfe281e304edd15c7c6f094f643c2b02149f36', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/brimmann2/squad-v2-sampled', endpoint='https://huggingface.co', repo_type='dataset', repo_id='brimmann2/squad-v2-sampled'), pr_revision=None, pr_num=None)

In [31]:
d["dev"][0]["embeddings"]

[1.6796875,
 -3.84375,
 -3.203125,
 -0.74609375,
 0.9609375,
 -5.5,
 -5.46875,
 6.625,
 8.1875,
 0.73828125,
 -3.28125,
 -2.734375,
 4.0,
 2.890625,
 -5.0625,
 -1.71875,
 0.36328125,
 6.03125,
 -1.96875,
 -5.0,
 0.318359375,
 -4.8125,
 4.71875,
 -0.322265625,
 -2.21875,
 6.03125,
 -0.99609375,
 -0.78125,
 -3.71875,
 -1.3359375,
 6.6875,
 -1.484375,
 4.875,
 5.15625,
 4.5,
 -2.828125,
 1.171875,
 -4.625,
 0.41796875,
 6.03125,
 5.875,
 -3.125,
 -0.048828125,
 -0.1123046875,
 1.3671875,
 -0.65625,
 1.9609375,
 11.5625,
 -6.1875,
 2.25,
 5.0625,
 -6.40625,
 0.578125,
 -10.3125,
 -0.267578125,
 -6.59375,
 3.09375,
 -6.09375,
 0.484375,
 -1.984375,
 -2.140625,
 4.09375,
 3.1875,
 -5.71875,
 1.703125,
 -9.5625,
 -8.5,
 -6.78125,
 0.26171875,
 -4.5,
 -8.375,
 -3.625,
 -4.1875,
 1.5703125,
 -0.5234375,
 -5.0625,
 -2.796875,
 4.34375,
 -3.1875,
 -5.40625,
 3.8125,
 2.734375,
 2.375,
 2.328125,
 2.015625,
 0.3828125,
 -4.25,
 1.5625,
 -3.046875,
 2.0,
 -4.9375,
 8.875,
 -1.046875,
 3.0,
 -0.8906

In [32]:
d["test"][0]["embeddings"]

[3.03125,
 4.1875,
 -0.9765625,
 -6.59375,
 -1.8125,
 4.8125,
 -3.390625,
 2.90625,
 -2.53125,
 3.390625,
 2.5625,
 3.34375,
 2.609375,
 -4.46875,
 -6.5625,
 -3.859375,
 2.109375,
 2.453125,
 4.75,
 -5.59375,
 -3.078125,
 -4.21875,
 1.4140625,
 0.95703125,
 -4.4375,
 -6.1875,
 0.369140625,
 -7.09375,
 -0.357421875,
 3.59375,
 6.03125,
 -3.921875,
 6.96875,
 -1.4453125,
 5.1875,
 -3.296875,
 -0.6328125,
 -0.306640625,
 -2.640625,
 -1.625,
 7.125,
 -5.5,
 2.203125,
 -2.140625,
 6.125,
 0.45703125,
 5.03125,
 10.6875,
 -6.1875,
 -4.40625,
 -6.625,
 -3.078125,
 -5.5625,
 -0.2890625,
 -0.73828125,
 -6.75,
 2.65625,
 -4.34375,
 -0.038818359375,
 -5.4375,
 -1.0,
 -0.55859375,
 10.0625,
 -5.21875,
 3.828125,
 -0.5859375,
 -8.8125,
 -4.0625,
 1.4375,
 -4.0625,
 -9.0,
 -0.173828125,
 -4.125,
 8.75,
 -0.90625,
 -4.375,
 -0.376953125,
 1.3515625,
 -2.1875,
 -5.59375,
 -2.125,
 -0.287109375,
 2.1875,
 6.0625,
 -3.203125,
 -4.375,
 -3.734375,
 -0.4296875,
 2.65625,
 -1.6328125,
 -9.375,
 5.46875,
 2

In [ ]:
import pickle
with open("embeds_test_list", "wb") as f:
    pickle.dump(all_embeds, f)

In [ ]:
updated_ds["context"]